In [ ]:
"""
=============================================================
FILE 45 — STATEFUL CONVERSATIONAL AGENTS
=============================================================

CONCEPTS TAUGHT
----------------
1. Stateful Conversations
2. Persistent Context
3. Conversational Memory
4. Thread Persistence
5. AI Personalization
6. Session Continuity
7. Multi-Turn Dialogue
8. Chat Memory Systems
9. Persistent Copilots
10. Enterprise Conversational AI

CORE IDEA
-----------
The AI remembers
past interactions across conversations.

FLOW
-----
Conversation
   ↓
Store State
   ↓
Retrieve State
   ↓
Context-Aware Responses

REAL WORLD USE CASES
---------------------
- AI copilots
- CRM assistants
- customer support AI
- enterprise chatbots
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict
from typing import Annotated

from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START, END

from langgraph.checkpoint.memory import MemorySaver

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE MODEL
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):

    messages: Annotated[list, add_messages]

# ============================================================
# STEP 5 — AGENT NODE
# ============================================================

def conversational_agent(state: State):

    response = llm.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

# ============================================================
# STEP 6 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "conversational_agent",
    conversational_agent
)

builder.add_edge(
    START,
    "conversational_agent"
)

builder.add_edge(
    "conversational_agent",
    END
)

# ============================================================
# STEP 7 — MEMORY CHECKPOINTER
# ============================================================

memory = MemorySaver()

# ============================================================
# STEP 8 — COMPILE GRAPH
# ============================================================

graph = builder.compile(
    checkpointer=memory
)

# ============================================================
# STEP 9 — VISUALIZE GRAPH
# ============================================================

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 10 — THREAD CONFIG
# ============================================================

config = {
    "configurable": {
        "thread_id": "enterprise_user_001"
    }
}

# ============================================================
# STEP 11 — FIRST CONVERSATION
# ============================================================

graph.invoke(
    {
        "messages":
        [
            (
                "human",
                """
                My name is Rahul.
                I work in healthcare analytics.
                """
            )
        ]
    },
    config=config
)

# ============================================================
# STEP 12 — SECOND CONVERSATION
# ============================================================

result = graph.invoke(
    {
        "messages":
        [
            (
                "human",
                """
                What do you remember about me?
                """
            )
        ]
    },
    config=config
)

# ============================================================
# STEP 13 — PRINT MEMORY RESPONSE
# ============================================================

print("\nMEMORY RESPONSE\n")
print("=" * 60)

for msg in result["messages"]:
    print(msg)